# Simple ARIMA Model for Non-Seasonal Time Series Forecasting

Our goal in this challenge is to apply the fundamental concepts of time series analysis to univariate data.

In these challenges we will follow these steps:
1. loading and visualizing the data;
2. training our models and making predictions;

##  1. Data Loading
Let's start by loading the time series we'll use in the challenge. Run the line below to download the dataset as a CSV file, then load the CSV into a DataFrame.

In [ ]:
!curl https://d32aokrjazspmn.cloudfront.net/materials/www_usage.csv > data/www_usage.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/www_usage.csv', names=['value'], header=0)
y = df.value

df.plot();

This abstract time series doesn't appear to be seasonal, but it shows an increasing trend and somewhat "sticky" (i.e., slightly auto-regressive) properties. Therefore it could be a good candidate for Auto-Regressive Moving Average (ARIMA) models.

## 2. Building an ARIMA Model
We will try to forecast the data using ARIMA models (Auto Regressive Integrated Moving Average).

To do this we will need to:
1. find how to make the time series stationary (the I in ARIMA)
2. find the auto-regressive (AR) part
3. find the moving average (MA) part
4. Fit
5. Evaluate performance

### Step 1 - Ensuring Stationarity

ARIMA models can only be applied to "stationary" time series.

👉 Check its stationarity definitively using the [`Augmented Dickey-Fuller test`](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html), especially the p-value

In [ ]:
# YOUR CODE HERE

For 95% confidence in stationarity, the p-value should be less than 0.05.
If the p-value is greater than 0.05, we cannot reject the null hypothesis (null hypothesis = "the process is not stationary").

If the time series is not stationary, it needs to be made stationary through **differencing**.
- This means taking the difference between each value and the previous value (*first difference*).
- If you want the *second difference*, repeat the operation on the differenced series, etc...

👉 Find the minimum differencing order needed to make it stationary (plot the curves to visualize and print the adfuller p-values to be sure)

<details>
    <summary>Hint</summary>

`pd.Series.diff`
</details>

In [ ]:
# YOUR CODE HERE

Here we have a close call between one and two differencing orders. Over-differencing time series can also degrade the performance of your ARIMA models. Let's take a closer look:

👉 Plot the autocorrelation chart ([`plot_acf`](https://www.statsmodels.org/stable/generated/statsmodels.graphics.tsaplots.plot_acf.html)) for both the 1st and 2nd differencing orders.

(💡Pro tip: avoid duplication of statsmodels charts by adding `;` at the end of each statsmodels plot instance or calling `plt.show()`)

In [ ]:
# YOUR CODE HERE

In our "second order difference" autocorrelation plot, the lag 1 coefficient is close to 0 and the second one extends into negative territory. This may indicate that we over-differenced the series. (Remember: we never care about lag 0 which is always equal to 1)

👉 Let's (temporarily) keep only one differencing order and call this series `y_diff` (we can always try more differences later)

In [ ]:
y_diff = y.diff().dropna()

We just found the "I" in ARIMA: `d = 1` for 1-difference before being stationary ("I" stands for "integration", "d" is for differencing...)

### Step 2 - Select the AR order (p) and MA order (q)

#### MA($\color{blue}{q}$) = number of lags where the $\color{blue}{ACF}$ of $Y^{\color{green}{(d)}}$ cuts off

The MA order (`q`) can be found by looking at the autocorrelation plot ([`plot_acf`](https://www.statsmodels.org/stable/generated/statsmodels.graphics.tsaplots.plot_acf.html)) applied to `y_diff`.

👉 Determine `q`

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
# YOUR CODE HERE

The maximum value we should evaluate for fitting our model seems to be q = 4. However, if we were to use Auto-ARIMA (more details on this later), we'd find that using q=2 gives ideal results, so let's try setting q=2 to start.

When in doubt, choose the simpler model that sufficiently explains Y.

#### AR($\color{red}{p}$) = number of lags where the $\color{red}{PACF}$ of $Y^{\color{green}{(d)}}$ cuts off

The AR order (`p`) can be found by examining the **p**artial autocorrelation plot [`plot_pacf`](https://www.statsmodels.org/stable/generated/statsmodels.graphics.tsaplots.plot_pacf.html) applied to `y_diff`.

(Partial autocorrelation can be thought of as the correlation between a series and its lag after excluding the contributions of intermediate lags. So PACF somehow conveys the "pure" correlation between a lag and the series)

👉 Determine `p`

In [ ]:
# YOUR CODE HERE

We can choose `p = 3` since the first 3 lag terms appear above the significance level, but we could also go with a simpler model with `p = 1`.

### Step 3 - Build the model

Now that you've selected the values of `p`, `d` and `q` for ARIMA,

👉 Build the `arima_model` from `statsmodels`.
- fits the model
- prints the model (`.summary`)

In [ ]:
# YOUR CODE HERE

☝️ If your p-values are too high, try removing those terms by reducing the corresponding AR or MA coefficients.

You can evaluate the overall performance of your fit by minimizing the [`AIC - Akaike Information Criterion`](https://medium.com/towardsdatascience/the-akaike-information-criterion-c20c8fd832f2) value.

It appears that (1,1,1) ARIMA models have less chance of overfitting (p-values remain low) and maintain a nearly similar AIC score compared to other models.

## 3. Evaluate Model Performance

👉 Visualize your model predictions with the `plot_predict()` method

- Look carefully at the default parameters of the method, especially the `dynamic` parameter.
- Do you think your model would actually perform this well?

In [ ]:
from statsmodels.graphics.tsaplots import plot_predict

In [ ]:
# YOUR CODE HERE

☝️ `dynamic=False` uses all available `y` values to predict `y_pred`, which makes your ARIMA forecast use up to $y_{t-1}$ to predict $y_t$. In reality, you don't have access to all `y`, especially if you want to forecast several intervals into the future.

👉 Try using `dynamic=True` to draw a forecast predicting the _last 15 values_ in a scenario where the model _only has access to data up to 85_. That is, the model:
- predicts 86 based on real [1...85]
- then predicts 87 based on its previously forecast value for 86 _plus_ [1...85]
- etc... iteratively up to 100

In [ ]:
# YOUR CODE HERE

☝️ This is still _not_ a _real_ forecast! Why?

<details>
    <summary>Answer</summary>

Our model "saw" the full `y_true` series during the fitting stage!
</details>

### 3.1 Out-of-Sample Forecasts (true "future")

👉 Create a train-test split by keeping only the last 15 data points for the test set, and train your ARIMA only on the training set.

In [ ]:
# YOUR CODE HERE

👉 Now we're at step 85 and we've never seen the future:
- Use the `get_forecast()` method on the `arima` model you fitted to "forecast" the next 15 data points (i.e., beyond the end of your training dataset)

The method returns a `PredictionResultsWrapper` object from `statsmodels`.

**💻 Store this result in a variable named `forecast_results`.**

It can be tricky to navigate at first, but here are some hints:
- You can find your predictions in `forecast_results.predicted_mean`
- Your confidence intervals are given by `forecast_results.conf_int()`

In [ ]:
# YOUR CODE HERE

👉 Plot the predicted values and also the upper and lower bounds of the 95% uncertainty interval

👉 Also try plotting your previous 85 `y` true data points to better understand the model's historical performance

In [ ]:
# YOUR CODE HERE

### 3.2 Can you trust your 95% confidence interval? (inference conditions)

👉 Plot the residuals `model.resid` to make sure there are no patterns
- Normal distribution
- Zero mean
- Uniform variance
- No auto-regressive pattern (you can plot_acf the residuals if you want)

Note: residuals are generated by "seeing" all data as in `plot_predict(dynamic=False)`

Also, try plotting a histogram or KDE fit of the residuals to see if the residuals are approximately normally distributed.

In [ ]:
# YOUR CODE HERE

## 3.3 Cross-validated performance metrics

👉 Below are the most common performance metrics for time series

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import acf

def forecast_accuracy(y_pred: pd.Series, y_true: pd.Series) -> float:

    mape = np.mean(np.abs(y_pred - y_true)/np.abs(y_true))  # Mean Absolute Percentage Error
    me = np.mean(y_pred - y_true)             # ME
    mae = np.mean(np.abs(y_pred - y_true))    # MAE
    mpe = np.mean((y_pred - y_true)/y_true)   # MPE
    rmse = np.mean((y_pred - y_true)**2)**.5  # RMSE
    corr = np.corrcoef(y_pred, y_true)[0,1]   # Correlation between the Actual and the Forecast
    mins = np.amin(np.hstack([y_pred.values.reshape(-1,1), y_true.values.reshape(-1,1)]), axis=1)
    maxs = np.amax(np.hstack([y_pred.values.reshape(-1,1), y_true.values.reshape(-1,1)]), axis=1)
    minmax = 1 - np.mean(mins/maxs)             # minmax
    acf1 = acf(y_pred-y_true, fft=False)[1]                      # Lag 1 Autocorrelation of Error

    forecast = ({
        'mape':mape,
        'me':me,
        'mae': mae,
        'mpe': mpe,
        'rmse':rmse,
        'acf1':acf1,
        'corr':corr,
        'minmax':minmax
    })

    return forecast

👉 Play with your ARIMA hyperparameters and see the effect on your forecast performance

In [ ]:
# YOUR CODE HERE

## 4 Grid Search

Try running a Grid Search for (p,d,q) using `pmdarima`. Use at least:
- `test='adf'`
- `trace=True`
- `error_action='ignore'`
- `suppress_warnings=True`

In [ ]:
import pmdarima as pm

model = pm.auto_arima(
    y_train,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=None,           # let model determine 'd'
    test='adf',       # using adf test to find optimal 'd'
    trace=True, error_action='ignore',  suppress_warnings=True
)

print(model.summary())

## (Optional) Cross-Validate Your Model's Performance

In practice, results and Grid Search should always be cross-validated:

Feel free to use [`sklearn.model_selection.TimeSeriesSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html) to create contiguous K-folds to really evaluate your model's performance and find the best hyperparameters after cross-validation.

<img src='https://scikit-learn.org/stable/_images/sphx_glr_plot_cv_indices_013.png'>

**ARIMA - Cross-Validation using TimeSeriesSplit + Grid Search**

In [ ]:
# YOUR CODE HERE

In [ ]:
df.sort_values('AIC').groupby('(p, d, q)').mean()['AIC'].sort_values()

☝️ Our initial model selection (1, 1, 2) is not too bad!
Note that the dataset (100 data points) is actually too small to cross-validate anything!